# Переобучение веток count_swap

Подключите датасет соревнования, private dataset из clevr_count_tv.zip и ранее загруженный clevr_verifier.zip или clevr_kaggle_inputs.zip для весов SigLIP. Выберите GPU T4 x2 и выполните Run All.

Прогон продолжает обучение ResNet и U-Net на train + val, переобучает геометрию и рассчитывает заново прогноз базовых детекторов и SigLIP. Веса union сохраняются, поскольку уже обучены на train + val. Формула ансамбля прежняя.

swap_tv.csv включает перестановку ответов. spatial_tv.csv сохраняет оценки перед ней. REFIT_UPSTREAM = False запускает сокращённое сравнение на сохранённой базе.

In [ ]:
import time

started = time.monotonic()
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
from concurrent.futures import ThreadPoolExecutor
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, display

REFIT_UPSTREAM = True
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
assert torch.cuda.device_count() == 2, "Settings → Accelerator → GPU T4 x2"
scratch = Path("/kaggle/temp")
ROOT = Path(tempfile.mkdtemp(prefix="clevr_refit_", dir=scratch if scratch.is_dir() else None))
LOGS = ROOT / "logs"
LOGS.mkdir()

def unpack(path, dest):
    with ZipFile(path) as source:
        for member in source.infolist():
            if not (dest / member.filename).resolve().is_relative_to(dest.resolve()):
                raise ValueError(member.filename)
        source.extractall(dest)

manifests = list(INPUT.rglob("count_tv_manifest.json"))
if not manifests:
    archives = list(INPUT.rglob("clevr_count_tv.zip"))
    if len(archives) != 1:
        raise FileNotFoundError("Подключите private dataset из clevr_count_tv.zip")
    unpack(archives[0], ROOT / "bundle")
    manifests = [ROOT / "bundle/count_tv_manifest.json"]
if len(manifests) != 1:
    raise ValueError("Подключите ровно одну версию clevr_refit")
package = manifests[0].parent
manifest = json.loads(manifests[0].read_text())
if manifest.get("version") != 2:
    raise ValueError("Обновите private dataset из новой версии clevr_count_tv.zip")
for name, expected in manifest["files"].items():
    with (package / name).open("rb") as handle:
        if hashlib.file_digest(handle, "sha256").hexdigest() != expected:
            raise ValueError(f"Повреждён файл {name}")

required = ("train.csv", "val.csv", "test.csv", "sample_submission.csv")
def datasets(parent):
    return [p.parent for p in parent.rglob("train.csv")
            if all((p.parent / name).is_file() for name in required)
            and all((p.parent / name).is_dir() for name in ("train", "val", "test"))]

data = datasets(INPUT)
if not data:
    extracted = ROOT / "data_zip"
    for i, path in enumerate(INPUT.rglob("*.zip")):
        with ZipFile(path) as archive:
            if set(required) <= {Path(name).name for name in archive.namelist()}:
                unpack(path, extracted / str(i))
    data = datasets(extracted)
if len(data) != 1:
    raise ValueError("Подключите данные соревнования")
DATA = data[0]
FIT = ROOT / "fit"
(FIT / "train").mkdir(parents=True)
frames = [pd.read_csv(DATA / f"{split}.csv") for split in ("train", "val")]
merged = pd.concat(frames, ignore_index=True)
if not merged.id.is_unique or not merged.label.isin([0, 1]).all():
    raise ValueError("Невалидные train/val")
with np.load(package / "fragment_train.npz") as bank:
    if not np.array_equal(bank["names"], merged.image.unique()):
        raise ValueError("Датасет не совпадает с банком обучающих фигур")
merged.to_csv(FIT / "train.csv", index=False)
for split, frame in zip(("train", "val"), frames):
    for name in frame.image.unique():
        source = DATA / split / name
        if not source.is_file():
            raise FileNotFoundError(source)
        (FIT / "train" / name).symlink_to(source)

env = dict(os.environ, CLEVR_PRIORS=str(package / "priors"), FRAGMENT_INPUT=str(package),
           PYTHONPATH=os.pathsep.join((str(package), str(package / "scripts/research"))),
           PYTHONUNBUFFERED="1", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2")
seeds = (421, 853)
times = {"prepare": time.monotonic() - started}
print("GPU", [torch.cuda.get_device_name(i) for i in range(2)], "fit scenes", merged.image.nunique())

def run(module, args, gpu, tag, data_root):
    command = [sys.executable, "-m", "scripts.research." + module, *map(str, args)]
    with (LOGS / f"{tag}.log").open("w") as log, subprocess.Popen(
            command, cwd=ROOT, env={**env, "CUDA_VISIBLE_DEVICES": str(gpu), "CLEVR_DATA": str(data_root)},
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            log.write(line)
            print(f"[{tag}] {line}", end="", flush=True)
        if process.wait():
            log.flush()
            shutil.copyfile(LOGS / f"{tag}.log", WORK / "refit_failed.log")
            raise RuntimeError(f"Ошибка {tag}. См. refit_failed.log")

def parallel(module, jobs, stage, data_root=DATA):
    tick = time.monotonic()
    with ThreadPoolExecutor(max_workers=2) as pool:
        tasks = [pool.submit(run, module, args, gpu, f"{stage}_{gpu}", data_root)
                 for gpu, args in enumerate(jobs)]
        for task in tasks:
            task.result()
    times[stage] = time.monotonic() - tick
    print(stage, round(times[stage], 1), "сек.", flush=True)

In [ ]:
def tasks(stage, shard=False):
    return [(stage,"--root",ROOT,"--seed",seed,*(("--shard",gpu) if shard else ()))
            for gpu,seed in enumerate(seeds)]

if REFIT_UPSTREAM:
    tick=time.monotonic()
    paths=json.loads((package/"siglip_inputs.json").read_text())
    for relative,digest in paths.items():
        name=Path(relative).name
        hits=[p for p in INPUT.rglob(name) if p.parent.name=="siglip2_vision"]
        destination=ROOT/relative
        destination.parent.mkdir(parents=True,exist_ok=True)
        if len(hits)==1:
            destination.symlink_to(hits[0])
        else:
            holders=[path for pattern in ("clevr_verifier.zip","clevr_kaggle_inputs.zip")
                     for path in INPUT.rglob(pattern)]
            if len(holders)!=1:
                raise ValueError("Подключите clevr_verifier.zip или clevr_kaggle_inputs.zip для весов SigLIP")
            with ZipFile(holders[0]) as source, source.open(relative) as f, destination.open("wb") as out:
                shutil.copyfileobj(f,out,length=2**20)
        with destination.open("rb") as handle:
            if hashlib.file_digest(handle,"sha256").hexdigest()!=digest:
                raise ValueError("Неверная версия весов SigLIP")
    weights=ROOT/"weights_in"
    (weights/"scene_silhouette.pt").symlink_to(package/"priors/scene_silhouette.pt")
    for seed in seeds:
        (weights/f"union_{seed}.pt").symlink_to(package/"initial"/f"union_{seed}.pt")
    env["CLEVR_WEIGHTS"]=str(weights)
    times["upstream_inputs"]=time.monotonic()-tick

parallel("count_swap_refit",tasks("fit"),"fit",FIT)

In [ ]:
if REFIT_UPSTREAM:
    module="swap_upstream_refit"
    def stage(name,args,fit=False,gpu=0):
        tick=time.monotonic()
        run(module,args,gpu,name,FIT if fit else DATA)
        times[name]=time.monotonic()-tick
    stage("init_upstream",("setup",),True,"")
    stage("train_regions",("prepare","--split","train"),True,"")
    parallel(module,[("fit","--arm",name) for name in ("r50_img","thaw")],"upstream_fit",FIT)
    stage("test_regions",("prepare","--split","test"),False,"")
    tick=time.monotonic()
    with ThreadPoolExecutor(2) as pool:
        left=pool.submit(run,module,("detect","--split","train","--arm","thaw"),0,"train_native",FIT)
        right=pool.submit(run,module,("detect","--split","test","--arm","thaw"),1,"test_native",DATA)
        left.result()
        right.result()
    times["native_tables"]=time.monotonic()-tick
    for split,data_root in (("train",FIT),("test",DATA)):
        parallel(module,[("crops","--split",split,"--shard",gpu) for gpu in (0,1)],f"crops_{split}",data_root)
    stage("fit_pool",("pool",),True)
    for split,data_root in (("train",FIT),("test",DATA)):
        parallel(module,[("encode","--split",split,"--shard",gpu) for gpu in (0,1)],f"encode_{split}",data_root)
    stage("fit_caption_head",("fit_crop",),True)
    parallel(module,[("detect","--arm","r50_img","--split","test"),("score_crop",)],"base_predictions")
    parallel(module,[("union","--shard",gpu) for gpu in (0,1)],"union_tables")
    stage("union_predictions",("combine",))
    env["CLEVR_PRIORS"]=str(ROOT/"result/weights")

In [ ]:
MODULE = "count_swap_refit"
def cpu(stage,data_root=DATA,extra=()):
    tick=time.monotonic()
    run(MODULE,(stage,"--root",ROOT,*extra),"",stage,data_root)
    times[stage]=time.monotonic()-tick

cpu("geometry",FIT)
parallel(MODULE,tasks("count"),"count_predictions")
cpu("anchor")
parallel(MODULE,tasks("objects",shard=True),"local_and_color")
cpu("typed")
parallel(MODULE,tasks("spatial",shard=True),"spatial_predictions")
cpu("export",extra=("--output",WORK/"swap_tv.csv","--without-swap",WORK/"spatial_tv.csv"))

In [ ]:
elapsed=time.monotonic()-started
report={"elapsed_minutes":elapsed/60,"within_50_minutes":elapsed<=50*60,
        "phase_seconds":times,"torch":torch.__version__,"cuda":torch.version.cuda,
        "gpu":[torch.cuda.get_device_name(i) for i in range(2)],
        "upstream_refit":REFIT_UPSTREAM}
(ROOT/"timing.json").write_text(json.dumps(report,indent=2))
with ZipFile(WORK/"swap_tv_results.zip","w",ZIP_DEFLATED,compresslevel=1) as archive:
    archive.write(WORK/"swap_tv.csv","swap_tv.csv")
    archive.write(WORK/"spatial_tv.csv","spatial_tv.csv")
    for path in (ROOT/"timing.json",ROOT/"spatial_geometry.pt",*LOGS.glob("*.log"),
                 *(ROOT/"curriculum").glob("*.json")):
        archive.write(path,path.relative_to(ROOT))
print(json.dumps(report,indent=2))
os.chdir(WORK)
for name in ("swap_tv.csv","spatial_tv.csv","swap_tv_results.zip"):
    display(FileLink(name))